In [1]:
import os
from arcpy.sa import *

In [7]:
import csv
import numpy as np

In [14]:
input_folder = r"C:\Users\olley\Documents\ArcGIS\Projects\SeagrassPredictions\xgboost_pred"
output_folder = r"C:\Users\olley\Documents\ArcGIS\Projects\SeagrassPredictions\xgboost_pred_classified"

In [15]:
classification_threshold = 0.24

In [16]:
# Check out Spatial Analyst
arcpy.CheckOutExtension("Spatial")

# Current ArcGIS Pro project
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap

In [18]:
for filename in os.listdir(input_folder):

    if filename.lower().endswith(".tif"):

        input_raster = os.path.join(input_folder, filename)

        # Add original raster to map
        input_layer = m.addDataFromPath(input_raster)

        # Apply Viridis
        sym = input_layer.symbology
        sym.updateColorizer("RasterStretchColorizer")

        for ramp in aprx.listColorRamps():
            if ramp.name.lower() == "viridis":
                sym.colorizer.colorRamp = ramp
                break

        input_layer.symbology = sym

        # Classification
        raster = Raster(input_raster)
        classified = Con(raster >= classification_threshold, 1, 0)

        output_name = os.path.splitext(filename)[0] + "_classified.tif"
        output_raster = os.path.join(output_folder, output_name)

        classified.save(output_raster)

In [19]:
results = []

for filename in os.listdir(output_folder):

    if filename.lower().endswith(".tif"):

        input_raster = os.path.join(output_folder, filename)

        # Add original raster to map
        input_layer = m.addDataFromPath(input_raster)

        # Apply Viridis
        sym = input_layer.symbology
        sym.updateColorizer("RasterStretchColorizer")

        for ramp in aprx.listColorRamps():
            if ramp.name.lower() == "viridis":
                sym.colorizer.colorRamp = ramp
                break

        input_layer.symbology = sym
        
        # calculate area
        
        raster = Raster(input_raster)
        
        arr = arcpy.RasterToNumPyArray(input_raster, nodata_to_value=0)
        
        pixel_count = np.sum(arr == 1)

        pixel_width = raster.meanCellWidth
        pixel_height = raster.meanCellHeight
        pixel_area = pixel_width * pixel_height
        
        total_area = pixel_count * pixel_area
        
        results.append([
            filename,
            pixel_count,
            pixel_area,
            total_area
        ])
        
        print(f"Area above threshold = {total_area:.2f}")

Area above threshold = 101654400.00
Area above threshold = 24762800.00
Area above threshold = 53702000.00
Area above threshold = 64814800.00
Area above threshold = 32830800.00
Area above threshold = 11177200.00
Area above threshold = 145210800.00
Area above threshold = 46446400.00
Area above threshold = 22643600.00
Area above threshold = 262956400.00
Area above threshold = 55741200.00
Area above threshold = 47612400.00
Area above threshold = 58624800.00
Area above threshold = 55996000.00
Area above threshold = 100014000.00
Area above threshold = 125066800.00
Area above threshold = 86050400.00
Area above threshold = 279049600.00
Area above threshold = 61927200.00
Area above threshold = 56118000.00
Area above threshold = 32510400.00
Area above threshold = 95512000.00
Area above threshold = 34989600.00
Area above threshold = 169775200.00
Area above threshold = 47424400.00


In [12]:
csv_output = os.path.join(output_folder, "threshold_areas.csv")

In [13]:
with open(csv_output, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "Raster",
        "Pixels_Above_Threshold",
        "Pixel_Area",
        "Total_Area"
    ])

    writer.writerows(results)

print(f"Results saved to: {csv_output}")

Results saved to: C:\Users\olley\Documents\ArcGIS\Projects\SeagrassPredictions\rf_pred_classified\threshold_areas.csv
